# 02 - Preprocessing

Goal: create a clean, balanced binary dataset for the first GAT experiment.

Decision for the first model:

- `Benign` -> `0`
- every other label -> `1`

We keep source and destination IP columns because the next step will build a graph: IP addresses are nodes, traffic flows are edges.

In [1]:
from argparse import Namespace
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocess_binary import (  # noqa: E402
    CATEGORICAL_COLUMNS,
    IP_COLUMNS,
    NUMERIC_COLUMNS,
    TARGET_COLUMN,
    make_binary_sample,
    save_splits,
    write_metadata,
)

DATA_PATH = PROJECT_ROOT / "data" / "iot23_combined_new.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Data exists:", DATA_PATH.exists())

Project root: c:\Users\Binh\OneDrive\Documents\CẦN NỘP\IDS_GAT_IOT23
Data exists: True


## Configuration

The first GAT experiment should be small enough to run on a laptop. Later we can increase `ROWS_PER_CLASS`.

In [2]:
ROWS_PER_CLASS = 50_000
CHUNK_SIZE = 500_000
RANDOM_STATE = 42

print("Rows per class:", ROWS_PER_CLASS)
print("Expected total rows:", ROWS_PER_CLASS * 2)

Rows per class: 50000
Expected total rows: 100000


## Create the binary sample

This cell scans the full CSV in chunks, cleans the values, and keeps a balanced random sample.

In [3]:
sample_df, summary = make_binary_sample(
    data_path=DATA_PATH,
    rows_per_class=ROWS_PER_CLASS,
    chunk_size=CHUNK_SIZE,
    random_state=RANDOM_STATE,
)

summary

{'total_rows_seen': 6046623,
 'requested_rows_per_class': 50000,
 'actual_rows': 100000,
 'binary_label_counts': {'0': 50000, '1': 50000},
 'raw_label_counts': {'Benign': 50000,
  'PartOfAHorizontalPortScan': 31606,
  'Okiru': 12293,
  'DDoS': 5921,
  'C&C': 162,
  'C&C-HeartBeat': 12,
  'Attack': 4,
  'C&C-Torii': 1,
  'C&C-FileDownload': 1}}

In [4]:
display(sample_df.head())
print("Shape:", sample_df.shape)
print("Binary label counts:")
display(sample_df[TARGET_COLUMN].value_counts().sort_index().to_frame("count"))
print("Raw label counts:")
display(sample_df["label"].value_counts().to_frame("count"))

,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label,binary_label
0,1.551385e+09,192.168.1.193,30535.0,197.73.143.161,8081.0,tcp,missing,0.000004,0.0,0.0,S0,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan,1
1,1.569018e+09,192.168.1.195,30011.0,162.248.88.215,62336.0,tcp,missing,0.000000,0.0,0.0,OTH,0.0,C,0.0,0.0,0.0,0.0,DDoS,1
2,1.545407e+09,192.168.1.196,33664.0,121.126.164.58,23.0,tcp,missing,0.000000,0.0,0.0,S0,0.0,S,1.0,60.0,0.0,0.0,Benign,0
3,1.552049e+09,192.168.1.197,63420.0,48.1.50.40,23.0,tcp,missing,0.000002,0.0,0.0,S0,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan,1
4,1.545397e+09,192.168.1.198,36097.0,83.89.62.111,37215.0,tcp,missing,0.000000,0.0,0.0,S0,0.0,S,1.0,40.0,0.0,0.0,Okiru,1


Shape: (100000, 19)
Binary label counts:


,count
binary_label,
0,50000
1,50000


Raw label counts:


,count
label,
Benign,50000
PartOfAHorizontalPortScan,31606
Okiru,12293
DDoS,5921
C&C,162
C&C-HeartBeat,12
Attack,4
C&C-Torii,1
C&C-FileDownload,1


## Save train, validation, and test files

We split after sampling and keep the binary labels balanced in each split.

In [5]:
paths = save_splits(
    sample=sample_df,
    output_dir=OUTPUT_DIR,
    random_state=RANDOM_STATE,
)

args = Namespace(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    rows_per_class=ROWS_PER_CLASS,
    chunk_size=CHUNK_SIZE,
    random_state=RANDOM_STATE,
)
write_metadata(OUTPUT_DIR, summary, paths, args)

paths

{'full': 'c:\\Users\\Binh\\OneDrive\\Documents\\CẦN NỘP\\IDS_GAT_IOT23\\data\\processed\\iot23_binary_sample.csv',
 'train': 'c:\\Users\\Binh\\OneDrive\\Documents\\CẦN NỘP\\IDS_GAT_IOT23\\data\\processed\\train.csv',
 'val': 'c:\\Users\\Binh\\OneDrive\\Documents\\CẦN NỘP\\IDS_GAT_IOT23\\data\\processed\\val.csv',
 'test': 'c:\\Users\\Binh\\OneDrive\\Documents\\CẦN NỘP\\IDS_GAT_IOT23\\data\\processed\\test.csv',
 'split_rows': {'train': 70000, 'val': 15000, 'test': 15000}}

In [6]:
train_df = pd.read_csv(OUTPUT_DIR / "train.csv")
val_df = pd.read_csv(OUTPUT_DIR / "val.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test.csv")

split_summary = pd.DataFrame({
    "rows": [len(train_df), len(val_df), len(test_df)],
    "benign": [
        (train_df[TARGET_COLUMN] == 0).sum(),
        (val_df[TARGET_COLUMN] == 0).sum(),
        (test_df[TARGET_COLUMN] == 0).sum(),
    ],
    "malicious": [
        (train_df[TARGET_COLUMN] == 1).sum(),
        (val_df[TARGET_COLUMN] == 1).sum(),
        (test_df[TARGET_COLUMN] == 1).sum(),
    ],
}, index=["train", "val", "test"])

split_summary

,rows,benign,malicious
train,70000,35000,35000
val,15000,7500,7500
test,15000,7500,7500


## Columns for the next step

The next notebook will turn these flows into a PyTorch Geometric graph.

In [ ]:
print("IP columns:", IP_COLUMNS)
print("Numeric columns:", NUMERIC_COLUMNS)
print("Categorical columns:", CATEGORICAL_COLUMNS)
print("Target column:", TARGET_COLUMN)

IP columns: ['id.orig_h', 'id.resp_h']
Numeric columns: ['ts', 'id.orig_p', 'id.resp_p', 'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes']
Categorical columns: ['proto', 'service', 'conn_state', 'history']
Target column: binary_label


: 